In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [2]:
def logistic(x):
    return 1 / (1 + np.exp(-x))

For this exercise, you'll be working with the [Titanic dataset](https://www.kaggle.com/c/titanic/data).  This dataset contains information about passengers on the Titanic, including their demographic details, ticket class, and other characteristics, as well as whether they survived the sinking of the ship. Your goal will be to use logistic regression to estimate the likelihood of survival based on these features.

In [3]:
titanic = pd.read_csv("../data/titanic.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


**Question 1:** Fit a logistic regression model to estimate the probability that a passenger survived based on their age?

What doe the model estimate the probability of survival for a 20 year old passenger? a 70 year old passenger?

In [19]:
titanic['Pclass'].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

In [7]:
titanic.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
titanic.shape

(891, 12)

In [9]:
survived_age_logreg = smf.logit("Survived ~ Age",
                          data = titanic).fit()

Optimization terminated successfully.
         Current function value: 0.672429
         Iterations 4


In [10]:
survived_age_logreg.params

Intercept   -0.056724
Age         -0.010963
dtype: float64

In [11]:
age = 20

logit_p = survived_age_logreg.params['Intercept'] + survived_age_logreg.params['Age']*age

print(f'Estimated Probability of Survival: {logistic(logit_p)}')

Estimated Probability of Survival: 0.4314364920047761


In [12]:
age = 70

logit_p = survived_age_logreg.params['Intercept'] + survived_age_logreg.params['Age']*age

print(f'Estimated Probability of Survival: {logistic(logit_p)}')

Estimated Probability of Survival: 0.30488018912276155


**Question 2:** Fit a new model that estimates the likelihood of survival based on the passenger's Sex.

How does the estimated probability of survival differ for male passengers compared to female passengers?

In [13]:
survived_sex_logreg = smf.logit("Survived ~ Sex",
                          data = titanic).fit()

Optimization terminated successfully.
         Current function value: 0.515041
         Iterations 5


In [14]:
survived_sex_logreg.params

Intercept      1.056589
Sex[T.male]   -2.513710
dtype: float64

In [15]:
male = 1

logit_p = survived_sex_logreg.params['Intercept'] + survived_sex_logreg.params['Sex[T.male]']*male

print(f'Estimated Probability of Survival for a Male: {logistic(logit_p)}')

Estimated Probability of Survival for a Male: 0.18890814558058938


In [16]:
male = 0

logit_p = survived_sex_logreg.params['Intercept'] + survived_sex_logreg.params['Sex[T.male]']*male

print(f'Estimated Probability of Survival for a Female: {logistic(logit_p)}')

Estimated Probability of Survival for a Female: 0.7420382165605097


In [18]:
print(f'Women were {0.7420382165605097/0.18890814558058938:.2f} times more likely to survive than men.')

Women were 3.93 times more likely to survive than men.


**Question 3:** Fit a new model to estimate the probability of survival using the age, sex, and passenger class. Make sure that your model treats the passenger class as a categorical variable. Recall that you can use the [patsy C() function](https://patsy.readthedocs.io/en/latest/categorical-coding.html) to indicate that a variable should be treated as a category.

What does the model estimate for the probability of a 20 year old female in first class surviving? What does it estimate for a 50 year old male in 3rd class?

In [20]:
survived_age_sex_class_logreg = smf.logit("Survived ~ Age + Sex + C(Pclass)",
                          data = titanic).fit()

Optimization terminated successfully.
         Current function value: 0.453279
         Iterations 6


In [21]:
survived_age_sex_class_logreg.params

Intercept         3.777013
Sex[T.male]      -2.522781
C(Pclass)[T.2]   -1.309799
C(Pclass)[T.3]   -2.580625
Age              -0.036985
dtype: float64

In [23]:
male = 0
age = 20
_class = 1

logit_p = (survived_age_sex_class_logreg.params['Intercept'] 
            + survived_age_sex_class_logreg.params['Sex[T.male]']*male
            + survived_age_sex_class_logreg.params['Age']*age)

print(f'Estimated Probability of Survival for a 20 year old Female in first class: {logistic(logit_p)}')

Estimated Probability of Survival for a 20 year old Female in first class: 0.954231374138041


In [24]:
male = 1
age = 50
_class = 3

logit_p = (survived_age_sex_class_logreg.params['Intercept'] 
            + survived_age_sex_class_logreg.params['Sex[T.male]']*male
            + survived_age_sex_class_logreg.params['Age']*age
            + survived_age_sex_class_logreg.params['C(Pclass)[T.3]']*_class)

print(f'Estimated Probability of Survival for a 50 year old Male in third class: {logistic(logit_p)}')

Estimated Probability of Survival for a 50 year old Male in third class: 0.000239454537475639
